In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/insurance.csv')
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
117,29,58.6,1.51,35.67,False,Chennai,retired,Low
183,30,88.9,1.75,75.19,True,Mysore,private_job,Medium
166,19,96.0,1.86,12.48,False,Mysore,freelancer,Low
7,36,67.8,1.65,50.21,False,Jalandhar,business_owner,Low
99,18,67.5,1.49,43.54,True,Bangalore,government_job,Medium


In [ ]:
#we create a copy cuz we will be applying a lot of feature engg steps.
df_feat = df.copy()

In [ ]:
#feature 1 : BMI
df_feat['bmi'] = df_feat['weight'] / (df_feat['height'] ** 2)

In [ ]:
#Feature 2: Age group
def age_group(age):
  if age < 25:
    return "young"
  elif age < 45:
    return "adult"
  elif age < 60:
    return "middle aged"
  else:
    return "senior"

In [ ]:
#Take every age in the age column, pass it through the age_group function
#and store the result in a new column called age_group.
df_feat['age_group'] = df['age'].apply(age_group)


In [ ]:
def lifestyle_risk(row):
  if row['smoker'] and row['bmi'] > 30:
    return "high"
  elif row['smoker'] and row['bmi'] > 27:
    return "medium"
  else:
    return "low"

In [ ]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis= 1)

In [ ]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = ["Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore", "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi", "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik", "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli", "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal", "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"]

In [ ]:
#Feature 4: City Tier
def city_tier(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else:
    return 3

In [ ]:
df_feat['city_tier'] = df_feat['city'].apply(city_tier)

In [ ]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']]

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
0,21.29,business_owner,31.234568,middle aged,high,2,High
1,37.14,private_job,25.944470,middle aged,low,1,Low
2,12.07,private_job,54.533333,adult,low,1,Low
3,76.42,government_job,42.214533,senior,low,2,Medium
4,49.28,private_job,22.793320,adult,low,1,Low
...,...,...,...,...,...,...,...
195,53.89,private_job,12.149901,adult,low,2,Medium
196,42.80,business_owner,18.142938,adult,low,1,Low
197,29.99,government_job,14.542088,middle aged,low,2,Low
198,70.42,private_job,13.102214,middle aged,low,1,Medium


In [ ]:
# Select features and target
x = df_feat[['bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'income_lpa', 'occupation']]
y = df_feat['insurance_premium_category']

In [ ]:
#define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]
#cuz we have to perform OHE on categorical features

In [ ]:
#create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers = [
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [ ]:
#create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", RandomForestClassifier(random_state= 42))
])

In [ ]:
#split data and train model
# x_train, x_test, y_train , y_test = train_test_split(x, y, test_size= 0.2, random_state= 1)
# pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [ ]:
# y_pred = pipeline.predict(x_test)
# accuracy_score(y_test, y_pred)

0.675

In [ ]:
x_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
97,27.296068,adult,low,1,10.36,business_owner
194,27.771203,senior,medium,2,65.54,retired
162,13.763418,middle aged,low,1,39.53,private_job
177,51.783979,middle aged,low,2,29.21,freelancer
38,22.722889,young,low,1,76.06,private_job


In [ ]:
import pickle
#save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
  pickle.dump(pipeline, f)

In [ ]:
import os

os.makedirs('/content/drive/MyDrive/ml_models', exist_ok=True)

In [ ]:
pickle_model_path = '/content/drive/MyDrive/ml_models/model.pkl'

with open(pickle_model_path, 'wb') as f:
    pickle.dump(pipeline, f)

with open('/content/drive/MyDrive/ml_models/model.pkl', 'rb') as f:
    pipeline = pickle.load(f)


In [ ]:
import sklearn
print(sklearn.__version__)

1.6.1
